## Setup & Dependencies
First, we'll install the necessary libraries and define our file paths.

In [ ]:
# !pip install transformers torch networkx tqdm

In [1]:
import json
import re
import unicodedata
import os
from collections import Counter, defaultdict
from pathlib import Path
from typing import Optional

import networkx as nx
from tqdm.auto import tqdm
from transformers import pipeline

# Configuration
BOOKS_DIR  = Path("books") 
OUTPUT_DIR = Path("data")
CHAPTERS_CACHE = OUTPUT_DIR / "chapters_cache.json"

NER_MODEL = "dslim/bert-base-NER"
MAX_TOKENS = 400 
STRIDE = 50 
COOC_WINDOW = 15 

STOP_NAMES = {
    "lord", "ser", "lady", "king", "queen", "prince", "princess",
    "maester", "septon", "father", "mother", "brother", "sister",
    "son", "daughter", "man", "woman", "boy", "girl", "old", "young",
    "first", "second", "hand", "night", "watch", "wall", "iron",
    "sept", "god", "gods",
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Chapter Extraction with Caching
This section handles the heavy lifting of reading the .txt files and splitting them by POV. I've added the logic to check if chapters_cache.json exists before running the regex.

In [2]:
CHAPTER_HEADING_RE = re.compile(r"^\s*([A-Z][A-Z\s]{1,30}?)\s*$", re.MULTILINE)

def get_chapters(force_refresh=False):
    """
    Loads chapters from cache if available, otherwise splits raw books and saves to cache.
    """
    if CHAPTERS_CACHE.exists() and not force_refresh:
        print(f"📦 Loading chapters from cache: {CHAPTERS_CACHE}")
        with open(CHAPTERS_CACHE, "r", encoding="utf-8") as f:
            return json.load(f)

    print("📖 Cache not found or refresh forced. Processing raw text files...")
    all_chapters = []
    paths = sorted(BOOKS_DIR.glob("*.txt"))
    
    if not paths:
        raise FileNotFoundError(f"No .txt files found in '{BOOKS_DIR}/'.")

    for path in paths:
        book_name = path.stem
        book_text = path.read_text(encoding="utf-8", errors="replace")
        matches = list(CHAPTER_HEADING_RE.finditer(book_text))
        
        if not matches:
            all_chapters.append({
                "book": book_name, "chapter_idx": 0, "heading": "UNKNOWN",
                "pov_char": "UNKNOWN", "text": book_text
            })
            continue

        for i, match in enumerate(matches):
            heading_raw = match.group(1).strip()
            body_start = match.end()
            body_end = matches[i + 1].start() if i + 1 < len(matches) else len(book_text)
            body = book_text[body_start:body_end].strip()

            if body:
                all_chapters.append({
                    "book": book_name,
                    "chapter_idx": i,
                    "heading": heading_raw,
                    "pov_char": heading_raw.title(),
                    "text": body,
                })

    # Save to cache
    with open(CHAPTERS_CACHE, "w", encoding="utf-8") as f:
        json.dump(all_chapters, f, indent=2, ensure_ascii=False)
    
    print(f"✅ Saved {len(all_chapters)} chapters to cache.")
    return all_chapters

# Execution
chapters_data = get_chapters()

📖 Cache not found or refresh forced. Processing raw text files...
✅ Saved 340 chapters to cache.


## NER Engine & Text Processing
Here we initialize the BERT model and define the normalization functions.

In [3]:
print(f"🤖 Loading NER model: {NER_MODEL}")
ner_pipeline = pipeline(
    task="ner",
    model=NER_MODEL,
    aggregation_strategy="simple",
    device=-1 # Set to 0 if you have a GPU
)

def normalise_name(raw: str) -> str:
    name = raw.replace("##", "")
    name = re.sub(r"\s+", " ", name)
    name = name.strip(" ,.'\"—-")
    name = unicodedata.normalize("NFC", name)
    return name.title()

def is_valid_name(name: str) -> bool:
    if len(name) < 2 or name.isdigit() or name.lower() in STOP_NAMES:
        return False
    return any(c.isalpha() for c in name)

def chunk_text(text, max_words=MAX_TOKENS, stride=STRIDE):
    words = text.split()
    chunks = []
    for i in range(0, len(words), max_words - stride):
        chunks.append(" ".join(words[i : i + max_words]))
        if i + max_words >= len(words): break
    return chunks

🤖 Loading NER model: dslim/bert-base-NER


Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


## Relationship Network Logic
This builds the co-occurrence graph based on sentence proximity.

In [4]:
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")

def build_network(chapter_text, persons, pov_char):
    G = nx.Graph()
    for char, count in persons.items():
        G.add_node(char, mentions=count, is_pov=(char == normalise_name(pov_char)))
    
    # Ensure POV is a node
    pov_name = normalise_name(pov_char)
    if is_valid_name(pov_name) and pov_name not in G:
        G.add_node(pov_name, mentions=0, is_pov=True)

    # Co-occurrence matching
    sentences = SENTENCE_SPLIT_RE.split(chapter_text)
    char_list = list(G.nodes)
    sentence_chars = []
    
    for sent in sentences:
        found = {name for name in char_list if name in sent}
        sentence_chars.append(found)

    for i in range(len(sentences)):
        window = set().union(*sentence_chars[i : i + COOC_WINDOW])
        window_chars = list(window)
        for idx_a in range(len(window_chars)):
            for idx_b in range(idx_a + 1, len(window_chars)):
                u, v = window_chars[idx_a], window_chars[idx_b]
                if G.has_edge(u, v): G[u][v]["weight"] += 1
                else: G.add_edge(u, v, weight=1)
    return G

## Run Pipeline & Export
The final execution loop. Note that since we loaded the chapters from the cache in Cell 2, this loop is now very clean.

In [ ]:
all_chapters_networks = []
global_char_data = defaultdict(lambda: {"total_mentions": 0, "chapter_mentions": {}, "books": []})

for chapter in tqdm(chapters_data, desc="Building Networks"):
    ch_label = f"{chapter['book']}::{chapter['heading']}"
    
    # Run NER
    chapter_persons = Counter()
    chunks = chunk_text(chapter["text"])
    for chunk in chunks:
        ents = ner_pipeline(chunk)
        for ent in ents:
            if ent["entity_group"].startswith("PER"):
                name = normalise_name(ent["word"])
                if is_valid_name(name):
                    chapter_persons[name] += 1
    
    # Build Graph
    G = build_network(chapter["text"], chapter_persons, chapter["pov_char"])
    
    # Update Global Stats
    for char, count in chapter_persons.items():
        global_char_data[char]["total_mentions"] += count
        global_char_data[char]["chapter_mentions"][ch_label] = count
        if chapter["book"] not in global_char_data[char]["books"]:
            global_char_data[char]["books"].append(chapter["book"])
            
    # Save chapter record
    all_chapters_networks.append({
        "book": chapter["book"],
        "chapter_idx": chapter["chapter_idx"],
        "heading": chapter["heading"],
        "pov_char": chapter["pov_char"],
        "network": {
            "nodes": [{"id": n, **d} for n, d in G.nodes(data=True)],
            "edges": [{"source": u, "target": v, **d} for u, v, d in G.edges(data=True)]
        }
    })

# Final Export
with open(OUTPUT_DIR / "characters.json", "w") as f:
    json.dump(global_char_data, f, indent=2)
with open(OUTPUT_DIR / "networks.json", "w") as f:
    json.dump(all_chapters_networks, f, indent=2)

print("✨ Analysis Complete!")

Building Networks:   0%|          | 0/340 [00:00<?, ?it/s]

## Export Chapter Networks to PDF
This cell loads the processed data, iterates through the chapters, and creates a visual graph for each.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# Configuration for Visualization
PLOTS_DIR = OUTPUT_DIR / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

MAX_PDFS = 10  # Limit to the first 10 chapters to save time/space (Set to None for all)

def export_networks_to_pdf(networks_file, limit=MAX_PDFS):
    with open(networks_file, "r", encoding="utf-8") as f:
        chapters = json.load(f)

    print(f"🎨 Generating PDFs in {PLOTS_DIR}...")
    
    # Process only up to the limit
    chapters_to_process = chapters[:limit] if limit else chapters

    for ch in tqdm(chapters_to_process, desc="Plotting Chapters"):
        net = ch["network"]
        book = ch["book"]
        heading = ch["heading"]
        
        # 1. Create NetworkX graph from the JSON data
        G = nx.Graph()
        for node in net["nodes"]:
            G.add_node(node["id"], mentions=node["mentions"], is_pov=node["is_pov"])
        for edge in net["edges"]:
            G.add_edge(edge["source"], edge["target"], weight=edge["weight"])

        if G.number_of_nodes() == 0:
            continue

        # 2. Setup Plot
        plt.figure(figsize=(12, 12))
        plt.title(f"Network: {book} - Chapter: {heading}\nPOV: {ch['pov_char']}", fontsize=16)

        # 3. Layout and Styling
        pos = nx.spring_layout(G, k=0.5, iterations=50) # Spreads nodes out
        
        # Node size based on mentions (with a minimum size)
        node_sizes = [max(300, d['mentions'] * 100) for n, d in G.nodes(data=True)]
        
        # Color POV character differently
        node_colors = ['#ff7675' if d['is_pov'] else '#74b9ff' for n, d in G.nodes(data=True)]
        
        # Edge width based on co-occurrence weight
        edge_widths = [max(1, d['weight'] / 2) for u, v, d in G.edges(data=True)]

        # 4. Draw
        nx.draw_networkx_nodes(G, pos, node_size=node_sizes, node_color=node_colors, alpha=0.8)
        nx.draw_networkx_labels(G, pos, font_size=10, font_family="sans-serif", font_weight="bold")
        nx.draw_networkx_edges(G, pos, width=edge_widths, edge_color="#b2bec3", alpha=0.5)

        # 5. Save to PDF
        pdf_path = PLOTS_DIR / f"{book}_{ch['chapter_idx']:02d}_{heading}.pdf"
        plt.axis('off')
        plt.savefig(pdf_path, format="pdf", bbox_inches='tight')
        plt.close() # Close plot to free up memory

export_networks_to_pdf(OUTPUT_DIR / "networks.json")